In [1]:
import time
import numpy as np
import sklearn
from sklearn.neighbors import KDTree, BallTree, radius_neighbors_graph
from sklearn.metrics.pairwise import euclidean_distances
from scipy.sparse import csr_array
from scipy.io import mmwrite, mmread

import cppimport
import cppimport.import_hook
from extensions.brute_force import *

class BruteForce(object):

    def __init__(self, points):
        self.points = points
        self.n = self.points.shape[0]
        self.d = self.points.shape[1]
    
    def query(self, X, k=1, return_distance=True, num_threads=4):
        assert X.shape[1] == self.d
        m = X.shape[0]
        dists = np.empty((m,k), dtype=np.float64)
        inds = np.empty((m,k), dtype=np.int64)
        brute_force_query(self.points, X, dists, inds, num_threads)
        if return_distance: return dists, inds
        else: return inds
    
    def query_radius_counts(self, X, r, num_threads=4):
        assert X.shape[1] == self.d
        return brute_force_query_radius_count_only(self.points, X, r, num_threads)
    
    def radius_neighbors_graph(self, radius, num_threads=4):
        rowptrs, colids, dists = brute_force_query_radius_neighbors(self.points, self.points, radius, num_threads)
        return csr_array((dists, colids, rowptrs), shape=(self.n,self.n))

        

n, d = 10000, 16
points = np.random.uniform(0,1,size=(n,d))

brute_force = BruteForce(points)
tree = KDTree(points)


In [5]:
%%timeit
graph = brute_force.radius_neighbors_graph(0.7, num_threads=8)

60.2 ms ± 244 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
graph1 = brute_force.radius_neighbors_graph(0.7, num_threads=1)
graph2 = brute_force.radius_neighbors_graph(0.7, num_threads=8)

In [8]:
print(np.allclose(graph1.todense(), graph2.todense()))

True
